# Limpieza del dataset de hospitales

Este notebook transforma el CSV original, que trae una fila por especialidad, en una tabla por hospital pensada para el prototipo de ambulancias.

Objetivos:
- conservar las especialidades de cada hospital en una sola fila
- normalizar nombres de columnas y valores vacíos
- generar coordenadas útiles para geolocalización
- añadir campos que ayuden a filtrar hospitales por síntomas o gravedad

In [7]:
from pathlib import Path

import pandas as pd
from pyproj import Transformer

NOTEBOOK_CWD = Path.cwd().resolve()
RAW_FILENAME = "centros_servicios_establecimientos_sanitarios.csv"

PROJECT_DIR = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD / "analisis_datos", NOTEBOOK_CWD, NOTEBOOK_CWD.parent]
        if (candidate / "data" / "raw" / RAW_FILENAME).exists()
    ),
    NOTEBOOK_CWD,
)

RAW_PATH = PROJECT_DIR / "data" / "raw" / RAW_FILENAME
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_PATH = PROCESSED_DIR / "centros_servicios_establecimientos_sanitarios_limpio.csv"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

Antes de iniciar, es importante revisar el dataset original para entender su estructura y contenido. Esto nos permitirá identificar las columnas relevantes y planificar cómo consolidar la información de las especialidades en una sola fila por hospital.

In [8]:
data = pd.read_csv(RAW_PATH, sep=";", dtype=str, keep_default_na=False)
data

,centro_nro_registro,centro_tipo,dependecia_funcional,dependecia_patrimonial,oferta_asistecial,municipio_nombre,direccion_vial_tipo,direccion_vial_nombre,direccion_vial_nro,direccion_informacon_adicional,direccion_codigo_postal,localizacion_coordenada_x,localizacion_coordenada_y
0,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Obtención de muestras,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720
1,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Otras unidades asistenciales,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720
2,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Urología,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720
3,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Radiodiagnóstico,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720
4,CH0001,Hospital especializado,Mutua de Accidentes de Trabajo,Mútua Patronal,Atención Continuada en Atención Primaria,Majadahonda,CTRA,M-515 (POZUELO),61,EDIFICIO A-C-D,28220,427057,4478720
...,...,...,...,...,...,...,...,...,...,...,...,...,...
50744,SS0283,Servicio sanitario integrado en una organizaci...,Privado No Benéfico,Privados,Medicina general/de familia,"Molinos, Los",CALLE,San Nicolás,5,,28460,409407,4506779
50745,SS0283,Servicio sanitario integrado en una organizaci...,Privado No Benéfico,Privados,Atención sanitaria a drogodependientes,"Molinos, Los",CALLE,San Nicolás,5,,28460,409407,4506779
50746,SS0285,Servicio sanitario integrado en una organizaci...,Privado No Benéfico,Privados,Fisioterapia,Majadahonda,CALLE,Azafrán,4,,28222,426359,4478639
50747,SS0286,Servicio sanitario integrado en una organizaci...,Privado No Benéfico,Privados,Depósito de medicamentos,Madrid,CALLE,de Margarita de Parma,1,,28050,443994,4483253


In [9]:
data.centro_tipo.unique()

<ArrowStringArray>
[                                       'Hospital especializado',                               'Otros centros con internamiento',
                                              'Hospital general',                            'Hospital de media y larga estancia',
        'Hospital de salud mental y tratamiento de toxicomanías',                                            'Centro Polivalente',
                'Centro de interrupción voluntaria del embarazo',                                                'Clínica dental',
                                            'Centro de diálisis',                                         'Centro de diagnóstico',
                                               'Centro de salud',                                  'Otros centros especializados',
                                               'Consulta médica',                    'Consulta de otros profesionales sanitarios',
   'Otros proveedores de asistencia sanitaria sin internamiento'

In [10]:
data.oferta_asistecial.unique()

<ArrowStringArray>
[                                                                    'Obtención de muestras',
                                                              'Otras unidades asistenciales',
                                                                                  'Urología',
                                                                          'Radiodiagnóstico',
                                                  'Atención Continuada en Atención Primaria',
                                                                     'Tratamiento del dolor',
                                                        'Cirugía ortopédica y Traumatología',
                                                                 'Cirugía mayor ambulatoria',
                                                                                 'Logopedia',
                                                               'Medicina general/de familia',
 ...
                                    

In [11]:
data.dependecia_funcional.unique()

<ArrowStringArray>
[               'Mutua de Accidentes de Trabajo',                    'Privado Benéfico (Iglesia)',
                           'Privado No Benéfico',  'Servicios o Institutos de Salud de las CC.AA',
 'Otros CESS Públicos de dependencia Autonómica',                  'Administración Penitenciaria',
    'Otros CESS Públicos de dependencia estatal',                                     'Municipio',
                         'Otro Privado Benéfico',                    'Otra dependencia funcional',
                  'Privado Benéfico (Cruz Roja)',                                              '',
                           'Otros CESS Públicos',                 'Instituto de Salud Carlos III',
                         'Ministerio de Defensa']
Length: 15, dtype: str

Usando funciones auxiliares, se procesará cada hospital para agrupar sus especialidades y generar una tabla limpia y estructurada.

In [12]:
import unicodedata


def clean_value(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "null"}:
        return pd.NA
    return text


def normalize_text(text):
    if pd.isna(text):
        return ""
    normalized = unicodedata.normalize("NFKD", str(text))
    normalized = normalized.encode("ascii", "ignore").decode("ascii")
    return " ".join(normalized.lower().split())


def first_non_null(series):
    valid = series.dropna()
    return valid.iloc[0] if not valid.empty else pd.NA


def join_unique(series, separator=" | "):
    values = []
    for value in series:
        cleaned = clean_value(value)
        if pd.isna(cleaned):
            continue
        if cleaned not in values:
            values.append(cleaned)
    return separator.join(values)


def build_attention_profiles(especialidades_series):
    normalized_specialties = {normalize_text(value) for value in especialidades_series.dropna()}
    profiles = []

    profile_keywords = {
        "cardiovascular": {
            "cardiologia",
            "cirugia cardiaca",
            "angiologia y cirugia vascular",
        },
        "neurologia": {
            "neurologia",
            "neurocirugia",
            "neurofisiologia",
        },
        "trauma_ortopedia": {
            "cirugia ortopedica y traumatologia",
            "lesionados medulares",
        },
        "obstetricia_neonatal": {
            "enfermeria obstetrico-ginecologica (matrona)",
            "cuidados intensivos neonatales",
            "cuidados intermedios neonatales",
            "fecundacion in vitro",
            "ginecologia",
            "obstetricia",
        },
        "pediatria": {
            "cirugia pediatrica",
            "pediatria",
            "cuidados intensivos neonatales",
            "cuidados intermedios neonatales",
        },
        "salud_mental": {
            "psiquiatria",
        },
        "diagnostico": {
            "radiodiagnostico",
            "laboratorio clinico",
            "bioquimica clinica",
            "anatomia patologica",
        },
        "urgencias_criticas": {
            "anestesia y reanimacion",
            "atencion continuada en atencion primaria",
            "cuidados intensivos",
        },
        "rehabilitacion": {
            "fisioterapia",
            "rehabilitacion",
            "logopedia",
            "terapia ocupacional",
        },
        "cirugia": {
            "cirugia general y digestivo",
            "cirugia mayor ambulatoria",
            "cirugia menor ambulatoria",
            "cirugia maxilofacial",
            "cirugia plastica y reparadora",
            "cirugia toracica",
            "cirugia refractiva",
        },
        "oncologia": {
            "oncologia",
            "hematologia y hemoterapia",
        },
    }

    for profile_name, keywords in profile_keywords.items():
        if normalized_specialties.intersection(keywords):
            profiles.append(profile_name)

    return " | ".join(profiles)


def is_public_dependency(value):
    text = normalize_text(value)
    public_markers = [
        "servicio madrileno de salud",
        "comunidad autonoma",
        "administracion",
        "ministerio",
        "municipal",
        "carlos iii",
    ]
    return any(marker in text for marker in public_markers)


column_renames = {
    "centro_nro_registro": "centro_id",
    "centro_tipo": "centro_tipo",
    "dependecia_funcional": "dependencia_funcional",
    "dependecia_patrimonial": "dependencia_patrimonial",
    "oferta_asistecial": "especialidad",
    "municipio_nombre": "municipio",
    "direccion_vial_tipo": "tipo_via",
    "direccion_vial_nombre": "nombre_via",
    "direccion_vial_nro": "numero_via",
    "direccion_informacon_adicional": "info_adicional",
    "direccion_codigo_postal": "codigo_postal",
    "localizacion_coordenada_x": "utm_x",
    "localizacion_coordenada_y": "utm_y",
}

raw = pd.read_csv(RAW_PATH, sep=";", dtype=str, keep_default_na=False)
df = raw.rename(columns=column_renames).copy()
df = df.apply(lambda column: column.map(clean_value))

# Keep only public dependency centers to remove private hospitals/centers.
df = df[df["dependencia_patrimonial"].map(is_public_dependency)].copy()
rows_after_public_filter = len(df)

# Keep only hospital centers relevant for emergency referral.
allowed_center_types = {"Hospital general", "Hospital especializado"}
df = df[df["centro_tipo"].isin(allowed_center_types)].copy()

for column in ["utm_x", "utm_y"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

transformer = Transformer.from_crs(25830, 4326, always_xy=True)
valid_coordinates = df["utm_x"].notna() & df["utm_y"].notna()
longitudes, latitudes = transformer.transform(
    df.loc[valid_coordinates, "utm_x"].to_numpy(),
    df.loc[valid_coordinates, "utm_y"].to_numpy(),
)
df.loc[valid_coordinates, "lon"] = longitudes
df.loc[valid_coordinates, "lat"] = latitudes

df["direccion_completa"] = (
    df[["tipo_via", "nombre_via", "numero_via", "info_adicional", "municipio", "codigo_postal"]]
    .fillna("")
    .agg(lambda row: " ".join(value for value in row if value), axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

cleaned = (
    df.groupby("centro_id", as_index=False)
    .agg(
        centro_tipo=("centro_tipo", first_non_null),
        dependencia_funcional=("dependencia_funcional", first_non_null),
        dependencia_patrimonial=("dependencia_patrimonial", first_non_null),
        municipio=("municipio", first_non_null),
        tipo_via=("tipo_via", first_non_null),
        nombre_via=("nombre_via", first_non_null),
        numero_via=("numero_via", first_non_null),
        info_adicional=("info_adicional", first_non_null),
        codigo_postal=("codigo_postal", first_non_null),
        utm_x=("utm_x", first_non_null),
        utm_y=("utm_y", first_non_null),
        lon=("lon", first_non_null),
        lat=("lat", first_non_null),
        direccion_completa=("direccion_completa", first_non_null),
        especialidades_texto=("especialidad", join_unique),
        num_especialidades=("especialidad", lambda series: series.dropna().nunique()),
        perfiles_atencion=("especialidad", build_attention_profiles),
    )
)

cleaned = cleaned.sort_values(["municipio", "centro_id"], kind="stable").reset_index(drop=True)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(PROCESSED_PATH, sep=";", index=False, encoding="utf-8-sig")

print(f"Filas originales (raw): {len(raw):,}")
print(f"Filas tras quitar privados: {rows_after_public_filter:,}")
print(f"Filas tras filtrar hospitales objetivo: {len(df):,}")
print(f"Hospitales/centros públicos únicos: {len(cleaned):,}")
print(f"Hospitales con coordenadas: {cleaned['lat'].notna().sum():,}")
print(f"Especialidades distintas: {df['especialidad'].nunique():,}")
cleaned.head(10)

Filas originales (raw): 50,749
Filas tras quitar privados: 7,451
Filas tras filtrar hospitales objetivo: 1,786
Hospitales/centros públicos únicos: 27
Hospitales con coordenadas: 26
Especialidades distintas: 96


,centro_id,centro_tipo,dependencia_funcional,dependencia_patrimonial,municipio,tipo_via,nombre_via,numero_via,info_adicional,codigo_postal,utm_x,utm_y,lon,lat,direccion_completa,especialidades_texto,num_especialidades,perfiles_atencion
0,CH0044,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Alcalá de Henares,CTRA,de Meco,S/N,<NA>,28805,470511.0,4484529.0,-3.348075,40.510956,CTRA de Meco S/N Alcalá de Henares 28805,Recuperación de oocitos | Extracción de órgano...,80,cardiovascular | neurologia | trauma_ortopedia...
1,CH0080,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Alcorcón,CALLE,Budapest,1,<NA>,28922,429031.0,4466866.0,-3.83568,40.349333,CALLE Budapest 1 Alcorcón 28922,Obtención de tejidos | Reumatología | Bioquími...,71,cardiovascular | neurologia | trauma_ortopedia...
2,CH0099,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Aranjuez,AVDA,Amazonas Central,S/N,<NA>,28300,447881.0,4434225.0,-3.611084,40.056662,AVDA Amazonas Central S/N Aranjuez 28300,Ginecología | Urología | Oftalmología | Cirugí...,57,cardiovascular | neurologia | trauma_ortopedia...
3,CH0100,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Arganda del Rey,RONDA,del Sur,10,<NA>,28500,461132.0,4460780.0,-3.457328,40.296617,RONDA del Sur 10 Arganda del Rey 28500,Oncología | Dermatología | Cuidados paliativos...,54,cardiovascular | neurologia | trauma_ortopedia...
4,CH0106,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Collado Villalba,CTRA,de Alpedrete,"M-608, KM. 41",<NA>,28400,414788.0,4500551.0,-4.007905,40.651421,"CTRA de Alpedrete M-608, KM. 41 Collado Villal...",Nefrología | Cirugía maxilofacial | Medicina p...,68,cardiovascular | neurologia | trauma_ortopedia...
5,CH0096,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Coslada,AVDA,de Marie Curie,s/n,<NA>,28822,454782.0,4474309.0,-3.532999,40.418179,AVDA de Marie Curie s/n Coslada 28822,Medicina interna | Medicina preventiva | Cirug...,56,cardiovascular | neurologia | trauma_ortopedia...
6,CH0086,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Fuenlabrada,CMNO,del Molino,2,<NA>,28942,430770.0,4459818.0,-3.814442,40.285988,CMNO del Molino 2 Fuenlabrada 28942,Endocrinología | Aparato digestivo | Obtención...,69,cardiovascular | neurologia | trauma_ortopedia...
7,CH0075,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Getafe,CTRA,de Madrid-Toledo,"KM.12,500",<NA>,28905,437027.0,4462721.0,-3.741124,40.312634,"CTRA de Madrid-Toledo KM.12,500 Getafe 28905",Obtención de muestras | Quemados | Neurofisiol...,74,cardiovascular | neurologia | trauma_ortopedia...
8,CH0052,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Leganés,AVDA,Orellana,S/N,<NA>,28911,434644.0,4463630.0,-3.76926,40.32064,AVDA Orellana S/N Leganés 28911,Tratamiento del dolor | Banco de tejidos | Pla...,62,cardiovascular | neurologia | trauma_ortopedia...
9,CH0023,Hospital general,Servicios o Institutos de Salud de las CC.AA,Servicio Madrileño de Salud,Madrid,AVDA,de Córdoba,S/N,<NA>,28041,440837.0,4469773.0,-3.696942,40.376441,AVDA de Córdoba S/N Madrid 28041,Otorrinolaringología | Oftalmología | Neurolog...,88,cardiovascular | neurologia | trauma_ortopedia...


In [13]:
cleaned.shape

(27, 18)

## Qué deja listo esta limpieza

El fichero procesado queda en `analisis_datos/data/processed/centros_servicios_establecimientos_sanitarios_limpio.csv`.

Columnas clave para el prototipo:
- `centro_id`: identificador estable del centro
- `lat` y `lon`: coordenadas ya convertidas para geolocalización
- `direccion_completa`: dirección lista para mostrar o depurar
- `especialidades_texto`: lista completa de especialidades del hospital
- `perfiles_atencion`: categorías resumidas útiles para triaje y derivación
- `num_especialidades`: tamaño del catálogo asistencial del hospital